In [1]:
import requests
import json
import os
import time
from dotenv import load_dotenv
load_dotenv()

True

## TEMPLATES

In [2]:
PROMPT_TASK_SETUP = """
0. Click "Select Campaign" and verify if the {campaign_name} is already exists. If it is, please click "Select Campaign" and choose {campaign_name}. Skip the the step 1-4.
1. Click "Create New Campaign"
2. In the Campaign Name, please fill: {campaign_name}, leave Campaign Description empty.
3. Click "Create Campaign"
4. Click "Select Campaign" and choose {campaign_name}
5. In "Thread Management" section, click "Create" below the Thread Title

DONE TASK!!!. Return the following fields in JSON format: campaign_name, campaign_id, and thread_id
{{
    "campaign_name": "string",
    "campaign_id": "int",
    "thread_id": "string"
}}
"""

OUTPUT_MODEL_FIELDS_SETUP = {
    "type": "object",
    "properties": {
        "campaign_name": {
            "type": "string",
            "description": "campaign_name field"
        },
        "campaign_id": {
            "type": "integer",
            "description": "campaign_id should be a positive integer"
        },
        "thread_id": {
            "type": "string",
            "description": "thread_id field"
        }
    },
    "required": ["campaign_name", "campaign_id", "thread_id"]
}

PROMPT_TASK_CHAT = """
**Task Description:**
You are the Operator of a web app used to simulate user interaction for testing automated game campaign planning.

Your job is to simulate the user's message flow — NOT to generate any content or alter any text. You must copy and paste assistant responses *exactly word-by-word*, simulating how a user would reply via automation.

---

**Please do step by step:**
1. Call `get_system_message` with temp_params=None to get the system's message.
2. Call `call_user_simulator` with temp_params=None to get the user's simulated reply.
3. Call `click_element_by_index` to click on the input box.
4. Call `paste_from_clipboard` with temp_params=None.
5. Call `click_the_send_button` with temp_params=None to click on the send button.
6. Wait for the assistant to generate the next message. You maybe need to call `wait` function more than once.
7. Only move to the next step after the assistant has generated the next message else continue call `wait` function.
8. Call `scroll` function to scroll the page.
9. Repeat steps 1-8 for each interaction cycle.

**TRIGGER DONE TASK**:
IF you see user's response is: "DONE TASK. PLEASE EXIST!". This means **the task is DONE**

MAKE SURE FOLLOW THE RULES, NO YAPPING!!!.
"""

EXCLUDE_ACTIONS_SETUP = [
    "search_google",
    "go_back",
    "save_pdf",
    "switch_tab",
    "close_tab",
    "extract_content",
    "send_keys",
    "wait",
    "scroll",
    "paste_from_clipboard",
    "call_user_simulator",
    "get_system_message",
    "click_the_send_button"
]

EXCLUDE_ACTIONS_CHAT = [
    "search_google",
    "go_back",
    "input_text",
    "save_pdf",
    "switch_tab",
    "close_tab",
    "extract_content",
    "send_keys",
    "get_dropdown_options",
    "select_dropdown_options",
    "drag_drop",
    "get_drag_elements",
    "get_element_coordinates",
    "execute_drag_operation",
    "click_element",
    "click_element_by_text"
]


PROMPT_SIMULATOR = """
## ✅ Tester Simulation Prompt for MktAgent Evaluation

### 🧠 Role
You are simulating a **Tester** evaluating an AI assistant (the "System") for its ability to fulfill a specific content creation task.

---

### 🎯 Objective
{AIM}  
Clearly define the intended goal or outcome the system should achieve.

---

### 🧾 Context
{CONTEXT}  
If the system is ask for information, please use the context to answer the question. Initially, you did not send context to the system.
---

### 📊 Guide: How to Grade (Pass/Fail) This Test Case
{GRADING_GUIDE}  
Use these criteria to determine if the system's output meets the expectations of the task. Do not reveal or reference this guide during the test.

---

### 🔁 Max Retry
**max_retry = {MAX_RETRY}**  
This is the maximum number of interactions (turns) allowed before the test must conclude with a pass/fail decision.

---

### 📌 Tester Guidelines
- Stay focused only on the defined **Objective** and **Context**.
- Use the **Guide** to evaluate each response.
- All feedback should directly relate to how well the system moves toward achieving the Objective.
- Never mention the grading guide in the conversation.
- Terminate the test using the Final Evaluation Format after reaching **max_retry** or when the test goal is clearly achieved or failed.

---

### 💬 Response Format
For **each turn**, return a JSON object in this format:

```json
{{
  "response": "Your next instruction or reaction to the system.",
  "feedback": "Brief evaluation of the system’s last response."
}}

---

### 💬 Response Format
For **each turn**, return a JSON object in this format:

```json
{{
  "response": "Your next instruction or reaction to the system.",
  "feedback": "Brief evaluation of the system’s last response."
}}

### Final Evaluation Format
When the test is complete, submit this JSON block:
{{
  "response": "DONE TASK. PLEASE EXIT!",
  "grade": "Pass"  // or "Fail",
  "feedback": "The reason why the task is failed or passed.",
}}
"""

##

## CONFIG TASKS

In [3]:
URL_TEST = "http://localhost:8501/"

TASK_SETUP = {
    "name": "Creating Campaign Test Case",
    "prompt": PROMPT_TASK_SETUP,
    "max_steps": 20,
    "output_model_fields": OUTPUT_MODEL_FIELDS_SETUP,
    "exclude_actions": EXCLUDE_ACTIONS_SETUP,
    "llm_provider": "google",
    "llm_model": "gemini-2.0-flash",
    "llm_temperature": 0.0,
    "enable_memory": False,
    "memory_interval": 10,
    "initial_actions": [
        {"open_tab": {"url": URL_TEST}}
    ]
}

TASK_CHAT = {
    "name": "Chat Test Case",
    "prompt": PROMPT_TASK_CHAT,
    "max_steps": 40,
    "output_model_fields": None,
    "exclude_actions": EXCLUDE_ACTIONS_CHAT,
    "llm_provider":"google",
    "llm_model": "gemini-2.0-flash",
    "llm_temperature": 0.0,
    "enable_memory": True,
    "memory_interval": 10,
    "initial_actions": [],
    "use_vision_for_planner": True,  # Example of custom param
    "planner_interval": 1,           # Example of custom param
    "is_planner_reasoning": True,
    # Custom planner LLM configuration:
    "planner_llm": {
        "provider": "google",        # Can be different from main LLM
        "model": "gemini-2.0-flash",   # Different model for planner
        "temperature": 0.0           # Different temperature for planner
    },
}

## RUN BROWSER AGENT

In [4]:
# import pandas as pd

# csv_file = pd.read_csv("E:/Eldenring/Campaign_Cases_Data.csv")
# csv_file.head()

In [5]:
test_cases = [
    {
        "case_id": "TC001",
        "status": "pending",
        "aim": "Generate a compelling, market-ready game description for the store page.",
        "context": "Game Title: *Emberfall Chronicles* \nGenre: Fantasy Action RPG \nPlatform: PC and Xbox \nCore Concept: Players become 'Flamebinders' who wield fire magic in a kingdom torn by elemental war. \nGameplay: Third-person combat, open world, faction alignment, spell crafting. \nKey Features: Dynamic weather, dragon mounts, branching storylines. \nTarget Audience: Fans of Elden Ring and Skyrim. \nUSP: Real-time weather influences spell effects and combat outcomes.",
        "grading_guide": "Pass if the description is vivid, clearly describes gameplay mechanics and unique features (like dynamic weather and dragon mounts), and feels suitable for a game store page. Fail if it’s vague, too short, or doesn’t mention the USP.",
        "max_retry": 3
    },
    {
        "case_id": "TC002",
        "status": "pending",
        "aim": "Create 5 engaging marketing slogans for a mobile sci-fi PvP shooter.",
        "context": "Game Title: *Zero Horizon* \nGenre: Sci-Fi Shooter \nPlatform: iOS & Android \nCore Concept: Teams of mercenaries battle in low-gravity arenas on Mars. \nGameplay: 5v5 competitive PvP, jetpack mobility, power-ups, and weapon customization. \nTarget Audience: Mobile gamers aged 16–30 who enjoy fast-paced competitive games. \nUSP: Mid-match gravity flips and vertical firefights.",
        "grading_guide": "Pass if 3 out of 5 slogans are punchy, futuristic, and reference PvP, Mars, or gravity. Fail if slogans feel flat, generic, or could apply to any game.",
        "max_retry": 2
    },
    {
        "case_id": "TC003",
        "status": "pending",
        "aim": "Write a launch tweet for a cozy mobile farming sim that includes a CTA (call to action).",
        "context": "Game Title: *SunSprout Valley* \nGenre: Cozy Farming Simulator \nPlatform: iOS & Android \nCore Concept: Players grow enchanted crops, care for talking animals, and decorate their dream homestead. \nKey Features: Seasonal festivals, mini-games, pet customization. \nTarget Audience: Casual mobile gamers, especially women aged 20–35. \nTone: Wholesome, friendly, inviting. \nCTA Requirement: Encourage people to download or pre-register.",
        "grading_guide": "Pass if the tweet is friendly, clearly describes a unique or cute game element, and includes a CTA like 'Download now' or 'Pre-register today!'. Fail if it lacks clarity, warmth, or doesn’t include a CTA.",
        "max_retry": 3
    }
]


In [6]:
def create_payload(case_id):
    """Create task payload with prompts populated from CSV data"""
    # Find the case in the test_cases list instead of csv_file
    case = next((case for case in test_cases if case["case_id"] == case_id), None)
    if case is None:
        raise ValueError(f"Case ID {case_id} not found in test cases")
    
    # Replace placeholders in prompts with actual data from CSV
    setup_prompt = PROMPT_TASK_SETUP.format(campaign_name=case['case_id'])
    simulator_prompt = PROMPT_SIMULATOR.format(AIM=case['aim'], CONTEXT=case['context'], GRADING_GUIDE=case['grading_guide'], MAX_RETRY=case['max_retry'])
    
    TASK_SETUP['prompt'] = setup_prompt
    
    
    payload = {
        "tasks": [
            TASK_SETUP,
            TASK_CHAT
        ],
        "laminar_api_key": os.getenv("LAMINAR_API_KEY", ""),
        "laminar_base_url": os.getenv("LAMINAR_BASE_URL", ""),
        "laminar_http_port": int(os.getenv("LAMINAR_HTTP_PORT", "0") or 0),
        "laminar_grpc_port": int(os.getenv("LAMINAR_GRPC_PORT", "0") or 0),
        "session_id": f"test-session-id-{case_id}",
        "simulator_provider": "google",
        "simulator_model": "gemini-2.0-flash",
        "simulator_temperature": 0.0,
        "simulator_task": simulator_prompt,
        "custom_actions": []
    }
    
    return payload

In [7]:
# API endpoint
API_BASE_URL = "http://localhost:8081"

In [8]:
def run_test_case(case_id):
    """Run a test case with the specified case ID and return the results"""
    payload = create_payload(case_id)
    
    response = requests.post(f"{API_BASE_URL}/tasks/run", json=payload)
    
    # Print response
    print(f"Status code: {response.status_code}")
    print(f"Response: {response.json()}")
    
    if response.status_code == 200:
        # Extract task ID
        task_id = response.json()['data']["message"].split(": ")[1]
        print(f"Task ID: {task_id}")
        
        # Poll for results
        return poll_results(task_id)
    else:
        print(f"Failed to start task: {response.text}")
        return None


def poll_results(task_id):
    """Poll the API for task results and return the data"""
    
    print("Polling for task results...")
    max_attempts = 100
    attempts = 0
    
    while attempts < max_attempts:
        attempts += 1
        response = requests.get(f"{API_BASE_URL}/tasks/{task_id}")
        
        if response.status_code == 200:
            data = response.json()['data']
            status = data.get("status")
            
            print(f"Task status: {status}")
            
            if status == "completed":
                print("Task completed!")
                # print("Results:")
                # print(json.dumps(data.get("results"), indent=2))
                
                # Check for simulator interactions
                # simulator_interactions = data.get("simulator_interactions", [])
                # if simulator_interactions:
                #     print("\nUser Simulator Interactions:")
                #     print(json.dumps(simulator_interactions, indent=2))
                return data
            elif status == "failed":
                print("Task failed!")
                print("Error:")
                print(data.get("error"))
                return data
            elif status == "cancelled":
                print("Task was cancelled")
                return data
        
        # Wait before polling again
        time.sleep(5)
    
    print("Max polling attempts reached. Task may still be running.")
    return None

In [9]:
result = run_test_case("TC002")

Status code: 200
Response: {'data': {'message': 'Task started with ID: 4d747a2c-d91b-45f8-86e9-c9950a92d06f'}, 'message': 'Success'}
Task ID: 4d747a2c-d91b-45f8-86e9-c9950a92d06f
Polling for task results...
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running
Task status: running


ConnectionError: HTTPConnectionPool(host='localhost', port=8081): Max retries exceeded with url: /tasks/4d747a2c-d91b-45f8-86e9-c9950a92d06f (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001FA7422A690>: Failed to establish a new connection: [WinError 10061] No connection could be made because the target machine actively refused it'))

In [12]:
result.keys()

dict_keys(['status', 'results', 'history', 'simulator_interactions'])

In [24]:
result['simulator_interactions']

[{'system_message': 'No content found for this campaign.',
  'image_url': [],
  'user_response': 'Okay, write a launch tweet for a cozy mobile farming sim that includes a CTA.',
  'feedback': 'Clear instruction given to start the task.',
  'grade': None},
 {'system_message': "Can you provide some more details about the game to tailor the tweet? Specifically, information like the core concept, key features, and target audience would be helpful. Additionally, do you have any specific campaign objectives or unique selling propositions that you'd like to highlight in the tweet?",
  'image_url': [],
  'user_response': "It's called SunSprout Valley. You grow enchanted crops, care for talking animals, and decorate your dream homestead. There are seasonal festivals, mini-games, and pet customization. Target audience is casual mobile gamers, especially women aged 20–35. Make it wholesome and friendly. Encourage people to download or pre-register.",
  'feedback': 'Provided the requested details 

In [15]:
result['status'], result['results'], result['simulator_interactions']

('completed',
 [{'task_name': 'Creating Campaign Test Case',
   'result': {'campaign_name': 'TC003',
    'campaign_id': 2,
    'thread_id': '4ca540ec-8248-41f2-a033-bd3167f5f715'}},
  {'task_name': 'Chat Test Case', 'result': ''}],
 [{'system_message': 'No content found for this campaign.',
   'image_url': [],
   'user_response': 'Okay, write a launch tweet for a cozy mobile farming sim that includes a CTA.',
   'feedback': 'Clear instruction given to start the task.',
   'grade': None},
  {'system_message': "Can you provide some more details about the game to tailor the tweet? Specifically, information like the core concept, key features, and target audience would be helpful. Additionally, do you have any specific campaign objectives or unique selling propositions that you'd like to highlight in the tweet?",
   'image_url': [],
   'user_response': "It's called SunSprout Valley. You grow enchanted crops, care for talking animals, and decorate your dream homestead. There are seasonal fe

In [14]:
result['results']

[{'task_name': 'Creating Campaign Test Case',
  'result': {'campaign_name': 'TC003',
   'campaign_id': 2,
   'thread_id': '4ca540ec-8248-41f2-a033-bd3167f5f715'}},
 {'task_name': 'Chat Test Case', 'result': ''}]

In [10]:
result['simulator_interactions']

[{'system_message': 'No content found for this campaign.',
  'image_url': [],
  'user_response': 'Okay, write a launch tweet for a cozy mobile farming sim that includes a CTA.',
  'feedback': 'Clear instruction given to start the task.',
  'grade': None},
 {'system_message': "Can you provide some more details about the game to tailor the tweet? Specifically, information like the core concept, key features, and target audience would be helpful. Additionally, do you have any specific campaign objectives or unique selling propositions that you'd like to highlight in the tweet?",
  'image_url': [],
  'user_response': "It's called SunSprout Valley. You grow enchanted crops, care for talking animals, and decorate your dream homestead. There are seasonal festivals, mini-games, and pet customization. Target audience is casual mobile gamers, especially women aged 20–35. Make it wholesome and friendly. Encourage people to download or pre-register.",
  'feedback': 'Provided the requested details 

In [ ]:
result['results']